### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema).
2. A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's a sunny in {location}"

# Binding tools
model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I\'ll call the function with "Boston" as the location argument. Make sure the JSON is correctly formatted with the name and arguments. No other functions are needed here. Just a straightforward call to get_weather with Boston.\n', 'tool_calls': [{'id': 'g1ge9zdz4', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 154, 'total_tokens': 262, 'completion_time': 0.19548073, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.007619929, 'prompt_tokens_details': None, 'queue_time': 0.071143424, 'total_time': 0.203100659}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_dema

### Tool execution Loops

In [13]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny today! 🌞 Make sure to enjoy the nice weather if you're heading out.


In [14]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools provided. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'ganezpbb0', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.133198475, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006178985, 'prompt_tokens_details': None, 'queue_time': 0.06733789, 'total_time': 0.13937746}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_f561c185cf', 'service_tier': 'on_demand',